# Notebook 0: Data Pipeline & Feature Engineering

**Purpose**: Fetch historical OHLCV data for all 20 assets + USDINR exchange rate, compute technical indicators and ML/DL features, and save everything to Parquet files.

**Run this first** before any strategy notebook.

**Output**:
- `data/raw/{ticker}.parquet` — raw OHLCV data
- `data/usdinr.parquet` — forward-filled INR→USD exchange rates
- `data/features/{ticker}_features.parquet` — processed features + target labels

In [13]:
import yfinance as yf
import pandas as pd
import numpy as np
import os
import time
import warnings
warnings.filterwarnings('ignore')

from config import (
    ALL_TICKERS, EXCHANGE_RATE_TICKER, START_DATE, END_DATE,
    EMA_SHORT, EMA_LONG, RSI_PERIOD, BB_PERIOD, BB_STD,
    ATR_PERIOD, OBV_EMA_PERIOD, FORWARD_RETURN_DAYS,
    BUY_RETURN_THRESHOLD, SELL_RETURN_THRESHOLD,
    RAW_DIR, FEATURES_DIR, USDINR_PATH,
    FETCH_RETRIES, FETCH_RETRY_DELAY,
)
from utils import setup_logging, create_directories

logger = setup_logging('data_pipeline')
create_directories()
print('Setup complete.')

Setup complete.


## 1. Data Fetching Functions

In [14]:
def fetch_single_ticker(ticker, start, end, retries=FETCH_RETRIES):
    """
    Fetch OHLCV data for a single ticker from yfinance.
    Retries on failure. Returns DataFrame or None.
    """
    for attempt in range(retries):
        try:
            df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
            if df is not None and len(df) > 0:
                # flatten multi-level columns if present
                if isinstance(df.columns, pd.MultiIndex):
                    df.columns = df.columns.get_level_values(0)
                # keep only OHLCV
                df = df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
                df.index = pd.to_datetime(df.index)
                df = df.sort_index()
                df = df[~df.index.duplicated(keep='first')]
                logger.info(f'Fetched {ticker}: {len(df)} rows')
                return df
        except Exception as e:
            logger.warning(f'Attempt {attempt+1} failed for {ticker}: {e}')
            time.sleep(FETCH_RETRY_DELAY)

    logger.error(f'All {retries} attempts failed for {ticker}. Skipping.')
    return None


def fetch_and_cache(ticker, start, end):
    """
    Fetch data and cache to Parquet. If cache exists, fetch only new data
    and append. Deduplicates on date index.
    """
    # sanitize ticker for filename (replace special chars)
    safe_name = ticker.replace('=', '_').replace('-', '_')
    path = os.path.join(RAW_DIR, f'{safe_name}.parquet')

    if os.path.exists(path):
        # load existing, fetch only new dates
        existing = pd.read_parquet(path)
        last_date = existing.index.max()
        fetch_start = (last_date + pd.Timedelta(days=1)).strftime('%Y-%m-%d')

        if fetch_start < end:
            new_data = fetch_single_ticker(ticker, fetch_start, end)
            if new_data is not None and len(new_data) > 0:
                combined = pd.concat([existing, new_data])
                combined = combined[~combined.index.duplicated(keep='last')]
                combined = combined.sort_index()
                combined.to_parquet(path)
                logger.info(f'Updated cache for {ticker}: {len(combined)} rows total')
                return combined
        return existing
    else:
        # fresh fetch
        df = fetch_single_ticker(ticker, start, end)
        if df is not None:
            df.to_parquet(path)
            logger.info(f'Cached {ticker}: {len(df)} rows')
        return df


def fetch_exchange_rate(start, end):
    """
    Fetch USDINR exchange rate, forward-fill to cover every calendar day.
    Saves to data/usdinr.parquet.
    """
    raw = fetch_single_ticker(EXCHANGE_RATE_TICKER, start, end)
    if raw is None:
        raise RuntimeError('Failed to fetch USDINR exchange rate data.')

    # create complete calendar date index and forward-fill
    full_dates = pd.date_range(start=start, end=end, freq='D')
    rates = pd.DataFrame(index=full_dates)
    rates['rate'] = raw['Close'].reindex(full_dates)
    rates['rate'] = rates['rate'].ffill().bfill()  # bfill for the very first few days if needed
    rates.index.name = 'Date'

    rates.to_parquet(USDINR_PATH)
    logger.info(f'Saved exchange rates: {len(rates)} days, no NaN: {rates["rate"].isna().sum() == 0}')
    return rates

## 2. Fetch All Data

In [15]:
# fetch exchange rate first
print('Fetching USDINR exchange rate...')
exchange_rates = fetch_exchange_rate(START_DATE, END_DATE)
print(f'Exchange rates: {len(exchange_rates)} days, range {exchange_rates["rate"].min():.2f} - {exchange_rates["rate"].max():.2f}')

# fetch all asset data
raw_data = {}
failed_tickers = []

for i, ticker in enumerate(ALL_TICKERS):
    print(f'[{i+1}/{len(ALL_TICKERS)}] Fetching {ticker}...')
    df = fetch_and_cache(ticker, START_DATE, END_DATE)
    if df is not None and len(df) > 0:
        raw_data[ticker] = df
    else:
        failed_tickers.append(ticker)
    time.sleep(1)  # polite delay between requests

print(f'\nFetched {len(raw_data)} assets successfully.')
if failed_tickers:
    print(f'Failed: {failed_tickers}')

Fetching USDINR exchange rate...
Exchange rates: 2192 days, range 68.37 - 85.41
[1/20] Fetching AAPL...
[2/20] Fetching MSFT...
[3/20] Fetching GOOGL...
[4/20] Fetching AMZN...
[5/20] Fetching NVDA...
[6/20] Fetching META...
[7/20] Fetching TSLA...
[8/20] Fetching JPM...
[9/20] Fetching JNJ...
[10/20] Fetching V...
[11/20] Fetching RELIANCE.NS...
[12/20] Fetching TCS.NS...
[13/20] Fetching HDFCBANK.NS...
[14/20] Fetching INFY.NS...
[15/20] Fetching ICICIBANK.NS...
[16/20] Fetching BHARTIARTL.NS...
[17/20] Fetching BTC-USD...
[18/20] Fetching ETH-USD...
[19/20] Fetching SOL-USD...
[20/20] Fetching GC=F...

Fetched 20 assets successfully.


## 3. Data Validation

In [16]:
def validate_data(data_dict):
    """Run basic data quality checks on all fetched data."""
    print(f'{"Ticker":<20} {"Rows":>6} {"Start":>12} {"End":>12} {"NaN%":>6}')
    print('-' * 60)
    for ticker, df in data_dict.items():
        nan_pct = df[['Close']].isna().mean().values[0] * 100
        start = df.index.min().strftime('%Y-%m-%d')
        end = df.index.max().strftime('%Y-%m-%d')
        print(f'{ticker:<20} {len(df):>6} {start:>12} {end:>12} {nan_pct:>5.1f}%')

        # assertions
        assert df.index.is_monotonic_increasing, f'{ticker}: dates not sorted'
        assert not df.index.duplicated().any(), f'{ticker}: duplicate dates'
        assert df['Close'].isna().sum() == 0, f'{ticker}: NaN in Close prices'

validate_data(raw_data)
print('\nAll data validations passed.')

Ticker                 Rows        Start          End   NaN%
------------------------------------------------------------
AAPL                   1509   2019-01-02   2024-12-30   0.0%
MSFT                   1509   2019-01-02   2024-12-30   0.0%
GOOGL                  1509   2019-01-02   2024-12-30   0.0%
AMZN                   1509   2019-01-02   2024-12-30   0.0%
NVDA                   1509   2019-01-02   2024-12-30   0.0%
META                   1509   2019-01-02   2024-12-30   0.0%
TSLA                   1509   2019-01-02   2024-12-30   0.0%
JPM                    1509   2019-01-02   2024-12-30   0.0%
JNJ                    1509   2019-01-02   2024-12-30   0.0%
V                      1509   2019-01-02   2024-12-30   0.0%
RELIANCE.NS            1480   2019-01-01   2024-12-30   0.0%
TCS.NS                 1480   2019-01-01   2024-12-30   0.0%
HDFCBANK.NS            1480   2019-01-01   2024-12-30   0.0%
INFY.NS                1480   2019-01-01   2024-12-30   0.0%
ICICIBANK.NS           1

## 4. Date Alignment

Create a union of all trading dates. Forward-fill each asset's prices on days it didn't trade.

In [17]:
def align_dates(data_dict):
    """
    Align all assets to a common date index (union of all trading dates).
    Forward-fill prices for days when an asset didn't trade.
    """
    # build union of all dates
    all_dates = sorted(set().union(*[set(df.index) for df in data_dict.values()]))
    all_dates = pd.DatetimeIndex(all_dates)

    aligned = {}
    for ticker, df in data_dict.items():
        reindexed = df.reindex(all_dates)
        # forward-fill prices (asset holds when its market is closed)
        reindexed[['Open', 'High', 'Low', 'Close']] = reindexed[['Open', 'High', 'Low', 'Close']].ffill()
        # volume = 0 on non-trading days
        reindexed['Volume'] = reindexed['Volume'].fillna(0)
        # drop rows before the asset's first data point
        first_valid = df.index.min()
        reindexed = reindexed[reindexed.index >= first_valid]
        reindexed.index.name = 'Date'
        aligned[ticker] = reindexed

    print(f'Aligned to {len(all_dates)} total dates.')
    return aligned

aligned_data = align_dates(raw_data)

# quick check
for ticker in list(aligned_data.keys())[:3]:
    df = aligned_data[ticker]
    print(f'{ticker}: {len(df)} rows, NaN in Close: {df["Close"].isna().sum()}')

Aligned to 2191 total dates.
AAPL: 2190 rows, NaN in Close: 0
MSFT: 2190 rows, NaN in Close: 0
GOOGL: 2190 rows, NaN in Close: 0


## 5. Technical Indicator Computation

In [18]:
def compute_ema(series, period):
    """Exponential Moving Average."""
    return series.ewm(span=period, adjust=False).mean()


def compute_rsi(close, period=RSI_PERIOD):
    """Relative Strength Index using Wilder's smoothing."""
    delta = close.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = (-delta).where(delta < 0, 0.0)

    avg_gain = gain.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, min_periods=period, adjust=False).mean()

    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    return rsi


def compute_bollinger_bands(close, period=BB_PERIOD, std_dev=BB_STD):
    """Bollinger Bands: middle, upper, lower."""
    middle = close.rolling(window=period).mean()
    std = close.rolling(window=period).std()
    upper = middle + std_dev * std
    lower = middle - std_dev * std
    return upper, middle, lower


def compute_atr(high, low, close, period=ATR_PERIOD):
    """Average True Range."""
    prev_close = close.shift(1)
    tr1 = high - low
    tr2 = (high - prev_close).abs()
    tr3 = (low - prev_close).abs()
    true_range = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    return true_range.rolling(window=period).mean()


def compute_obv(close, volume):
    """On-Balance Volume."""
    direction = np.sign(close.diff()).fillna(0)
    obv = (direction * volume).cumsum()
    return obv

In [19]:
def compute_all_features(df):
    """
    Compute all technical indicators and ML/DL features for a single asset.
    Returns the DataFrame with all new columns added.
    """
    df = df.copy()
    close = df['Close']
    high = df['High']
    low = df['Low']
    volume = df['Volume']

    # --- Technical Indicators (raw, for technical strategy) ---
    df['ema_20'] = compute_ema(close, EMA_SHORT)
    df['ema_50'] = compute_ema(close, EMA_LONG)
    df['rsi_14'] = compute_rsi(close, RSI_PERIOD)
    df['bb_upper'], df['bb_middle'], df['bb_lower'] = compute_bollinger_bands(close, BB_PERIOD, BB_STD)
    df['atr_14'] = compute_atr(high, low, close, ATR_PERIOD)
    df['obv'] = compute_obv(close, volume)
    df['obv_ema_20'] = compute_ema(df['obv'], OBV_EMA_PERIOD)

    # --- ML/DL Features (normalized ratios) ---
    # log returns at different horizons
    df['return_1d'] = np.log(close / close.shift(1))
    df['return_5d'] = np.log(close / close.shift(5))
    df['return_10d'] = np.log(close / close.shift(10))
    df['return_20d'] = np.log(close / close.shift(20))

    # price-relative indicator ratios
    df['ema_20_ratio'] = df['ema_20'] / close
    df['ema_50_ratio'] = df['ema_50'] / close
    df['bb_upper_ratio'] = df['bb_upper'] / close
    df['bb_lower_ratio'] = df['bb_lower'] / close
    df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / df['bb_middle']
    df['atr_14_norm'] = df['atr_14'] / close

    # volume-based features
    df['obv_ema_ratio'] = df['obv'] / df['obv_ema_20'].replace(0, np.nan)
    volume_sma = volume.rolling(window=20).mean()
    df['volume_ratio'] = volume / volume_sma.replace(0, np.nan)

    # price structure features
    df['high_low_ratio'] = (high - low) / close
    df['close_open_ratio'] = (close - df['Open']) / df['Open'].replace(0, np.nan)

    # --- Target Label ---
    # forward 5-day return, shifted by 1 to prevent look-ahead
    forward_return = close.shift(-FORWARD_RETURN_DAYS) / close.shift(-1) - 1
    df['forward_return'] = forward_return

    # classify: 0=SELL, 1=HOLD, 2=BUY
    conditions = [
        forward_return > BUY_RETURN_THRESHOLD,
        forward_return < SELL_RETURN_THRESHOLD,
    ]
    choices = [2, 0]  # BUY=2, SELL=0
    df['target'] = np.select(conditions, choices, default=1)  # HOLD=1

    # set target to NaN where forward return is NaN (last few days)
    df.loc[forward_return.isna(), 'target'] = np.nan

    return df

## 6. Process All Assets

In [20]:
feature_data = {}

for i, (ticker, df) in enumerate(aligned_data.items()):
    print(f'[{i+1}/{len(aligned_data)}] Computing features for {ticker}...')
    featured = compute_all_features(df)
    feature_data[ticker] = featured

    # save to parquet
    safe_name = ticker.replace('=', '_').replace('-', '_')
    path = os.path.join(FEATURES_DIR, f'{safe_name}_features.parquet')
    featured.to_parquet(path)

print(f'\nAll {len(feature_data)} assets processed and saved.')

[1/20] Computing features for AAPL...
[2/20] Computing features for MSFT...
[3/20] Computing features for GOOGL...
[4/20] Computing features for AMZN...
[5/20] Computing features for NVDA...
[6/20] Computing features for META...
[7/20] Computing features for TSLA...
[8/20] Computing features for JPM...
[9/20] Computing features for JNJ...
[10/20] Computing features for V...
[11/20] Computing features for RELIANCE.NS...
[12/20] Computing features for TCS.NS...
[13/20] Computing features for HDFCBANK.NS...
[14/20] Computing features for INFY.NS...
[15/20] Computing features for ICICIBANK.NS...
[16/20] Computing features for BHARTIARTL.NS...
[17/20] Computing features for BTC-USD...
[18/20] Computing features for ETH-USD...
[19/20] Computing features for SOL-USD...
[20/20] Computing features for GC=F...

All 20 assets processed and saved.


## 7. Feature Validation

In [21]:
def validate_features(feature_dict):
    """Validate computed features across all assets."""
    from config import FEATURE_COLS

    print(f'{"Ticker":<20} {"Rows":>6} {"Valid":>6} {"Target Dist (S/H/B)":>25}')
    print('-' * 65)

    for ticker, df in feature_dict.items():
        valid_mask = df[FEATURE_COLS].notna().all(axis=1) & df['target'].notna()
        valid_rows = valid_mask.sum()

        # target distribution
        target_valid = df.loc[valid_mask, 'target']
        sell_pct = (target_valid == 0).mean() * 100
        hold_pct = (target_valid == 1).mean() * 100
        buy_pct = (target_valid == 2).mean() * 100

        print(f'{ticker:<20} {len(df):>6} {valid_rows:>6} {sell_pct:>7.1f}% / {hold_pct:.1f}% / {buy_pct:.1f}%')

validate_features(feature_data)

Ticker                 Rows  Valid       Target Dist (S/H/B)
-----------------------------------------------------------------
AAPL                   2190   2165    27.4% / 32.4% / 40.2%
MSFT                   2190   2165    27.7% / 34.2% / 38.1%
GOOGL                  2190   2165    29.9% / 30.0% / 40.1%
AMZN                   2190   2165    32.7% / 29.6% / 37.7%
NVDA                   2190   2165    34.5% / 17.6% / 47.9%
META                   2190   2165    33.5% / 25.4% / 41.1%
TSLA                   2190   2165    38.4% / 15.7% / 45.9%
JPM                    2190   2165    27.9% / 35.2% / 36.9%
JNJ                    2190   2165    25.2% / 47.1% / 27.7%
V                      2190   2165    25.6% / 40.1% / 34.3%
RELIANCE.NS            2191   2166    30.2% / 35.3% / 34.4%
TCS.NS                 2191   2166    28.6% / 38.6% / 32.8%
HDFCBANK.NS            2191   2166    27.4% / 40.3% / 32.3%
INFY.NS                2191   2166    27.1% / 36.0% / 36.9%
ICICIBANK.NS           2191   216

## 8. Determine Common Backtest Start Date

In [22]:
from utils import get_common_start_date

common_start = get_common_start_date(feature_data)
print(f'Common backtest start date: {common_start.strftime("%Y-%m-%d")}')
print(f'Warm-up period: {START_DATE} to {common_start.strftime("%Y-%m-%d")}')
print(f'Backtest period: {common_start.strftime("%Y-%m-%d")} to {END_DATE}')

# save this date so strategy notebooks can read it
pd.Series({'common_start_date': common_start.strftime('%Y-%m-%d')}).to_json(
    os.path.join('data', 'common_start_date.json')
)
print('Saved common start date to data/common_start_date.json')

Common backtest start date: 2022-04-30
Warm-up period: 2019-01-01 to 2022-04-30
Backtest period: 2022-04-30 to 2024-12-31
Saved common start date to data/common_start_date.json


## 9. Quick Data Summary

In [23]:
# summary of what was saved
print('=== Files Saved ===')
print(f'Exchange rates: {USDINR_PATH}')
print(f'Raw data: {RAW_DIR}/ ({len(os.listdir(RAW_DIR))} files)')
print(f'Features: {FEATURES_DIR}/ ({len(os.listdir(FEATURES_DIR))} files)')
print(f'Common start: data/common_start_date.json')

# show a sample
sample_ticker = 'AAPL'
safe = sample_ticker.replace('=', '_').replace('-', '_')
sample = pd.read_parquet(os.path.join(FEATURES_DIR, f'{safe}_features.parquet'))
print(f'\nSample ({sample_ticker}) columns: {list(sample.columns)}')
print(f'Shape: {sample.shape}')
sample.tail(3)

=== Files Saved ===
Exchange rates: data/usdinr.parquet
Raw data: data/raw/ (20 files)
Features: data/features/ (20 files)
Common start: data/common_start_date.json

Sample (AAPL) columns: ['Open', 'High', 'Low', 'Close', 'Volume', 'ema_20', 'ema_50', 'rsi_14', 'bb_upper', 'bb_middle', 'bb_lower', 'atr_14', 'obv', 'obv_ema_20', 'return_1d', 'return_5d', 'return_10d', 'return_20d', 'ema_20_ratio', 'ema_50_ratio', 'bb_upper_ratio', 'bb_lower_ratio', 'bb_width', 'atr_14_norm', 'obv_ema_ratio', 'volume_ratio', 'high_low_ratio', 'close_open_ratio', 'forward_return', 'target']
Shape: (2190, 30)


Price,Open,High,Low,Close,Volume,ema_20,ema_50,rsi_14,bb_upper,bb_middle,...,bb_upper_ratio,bb_lower_ratio,bb_width,atr_14_norm,obv_ema_ratio,volume_ratio,high_low_ratio,close_open_ratio,forward_return,target
Date,,,,,,,,,,,,,,,,,,,,,
2024-12-28,256.429175,257.294474,251.685102,254.201355,0.0,250.419575,242.457081,66.392865,259.120800,250.683592,...,1.019353,0.952970,0.067314,0.020217,1.023774,0.000000,0.022067,-0.008688,NaN,NaN
2024-12-29,256.429175,257.294474,251.685102,254.201355,0.0,250.779745,242.917641,66.392865,259.315976,251.123190,...,1.020120,0.955661,0.065249,0.020940,1.021461,0.000000,0.022067,-0.008688,NaN,NaN
2024-12-30,250.859609,252.122713,249.387654,250.829773,35557500.0,250.784509,243.227921,55.473223,259.235656,251.343485,...,1.033512,0.970584,0.062800,0.021536,1.014227,1.096464,0.010904,-0.000119,NaN,NaN


In [24]:
print('Notebook 0 complete. You can now run notebooks 1, 2, and 3 in any order.')

Notebook 0 complete. You can now run notebooks 1, 2, and 3 in any order.
